In [1]:
import sys
sys.path.append(r"C:\Users\zhossai3\Desktop\Uncertainty\imputation_Uncertainty")  # <<< updated path here to refer to the implementation
import Utils
import Inject_Missing_Values
import imputers
import Gain
import MIWAE


In [2]:
import numpy as np
import pandas as pd
import torch
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from Inject_Missing_Values import *
from imputers import *
from imputers_updated import *
from Utils import *
from Gain import *
from MIWAE import *
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer


In [3]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import time

In [4]:
# Set random seed
np.random.seed(42)
torch.manual_seed(42)
rng = np.random.default_rng(42)
import torch.nn as nn

Loading the data

In [5]:
from sklearn.datasets import fetch_california_housing
import pandas as pd

# Load dataset
df = fetch_california_housing(as_frame=True)

# Convert to DataFrame
df = df.frame

#dropping the target coloumn
data = df.drop(columns=["MedHouseVal"])
print(df.shape)
data.head()


(20640, 9)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


In [6]:
X_full = data.copy()

In [7]:
print(X_full.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   MedInc      20640 non-null  float64
 1   HouseAge    20640 non-null  float64
 2   AveRooms    20640 non-null  float64
 3   AveBedrms   20640 non-null  float64
 4   Population  20640 non-null  float64
 5   AveOccup    20640 non-null  float64
 6   Latitude    20640 non-null  float64
 7   Longitude   20640 non-null  float64
dtypes: float64(8)
memory usage: 1.3 MB
None


MinMax Scalling

In [8]:
#scaler = MinMaxScaler() #data [0,1]
#X_full = pd.DataFrame(scaler.fit_transform(X_full), columns=X_full.columns)

scaler = StandardScaler()
X_full =  pd.DataFrame(scaler.fit_transform(X_full), columns=X_full.columns)

Storing the clean data as Ground Truth

In [9]:
GroundTruth = X_full.copy()
X = X_full.copy()

In [10]:
print(GroundTruth)

         MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  \
0      2.344766  0.982143  0.628559  -0.153758   -0.974429 -0.049597   
1      2.332238 -0.607019  0.327041  -0.263336    0.861439 -0.092512   
2      1.782699  1.856182  1.155620  -0.049016   -0.820777 -0.025843   
3      0.932968  1.856182  0.156966  -0.049833   -0.766028 -0.050329   
4     -0.012881  1.856182  0.344711  -0.032906   -0.759847 -0.085616   
...         ...       ...       ...        ...         ...       ...   
20635 -1.216128 -0.289187 -0.155023   0.077354   -0.512592 -0.049110   
20636 -0.691593 -0.845393  0.276881   0.462365   -0.944405  0.005021   
20637 -1.142593 -0.924851 -0.090318   0.049414   -0.369537 -0.071735   
20638 -1.054583 -0.845393 -0.040211   0.158778   -0.604429 -0.091225   
20639 -0.780129 -1.004309 -0.070443   0.138403   -0.033977 -0.043682   

       Latitude  Longitude  
0      1.052548  -1.327835  
1      1.043185  -1.322844  
2      1.038503  -1.332827  
3      1.038503  -1

In [11]:
GroundTruth.nunique()

MedInc        12928
HouseAge         52
AveRooms      19392
AveBedrms     14233
Population     3888
AveOccup      18841
Latitude        862
Longitude       844
dtype: int64

Converting GroundTruth to tensor

In [12]:
GroundTruth_tensor = torch.tensor(GroundTruth.values, dtype=torch.double)

In [13]:
print("MedInc",GroundTruth['MedInc'].unique())
print("HouseAge",GroundTruth['HouseAge'].unique())
print("AveRooms",GroundTruth['AveRooms'].unique())
print("AveBedrms",GroundTruth['AveBedrms'].unique())
print("Population",GroundTruth["Population"].unique())
print("AveOccup",GroundTruth['AveOccup'].unique())
print("Latitude",GroundTruth['Latitude'].unique())
print("Longitude",GroundTruth['Longitude'].unique())

MedInc [ 2.34476576  2.33223796  1.7826994  ... -0.79528915 -0.79197297
 -0.93504249]
HouseAge [ 0.98214266 -0.60701891  1.85618152  1.06160074  1.69726537  0.90268458
  1.61780729  1.53834921  1.77672344  1.14105882 -2.11672241  1.37943305
 -0.20972852 -0.68647699 -0.92485123  0.58485227 -0.76593507 -0.44810276
  0.74376842  0.50539419 -1.48105778 -1.00430931 -0.13027044  0.8232265
  0.18756187  0.02864572 -0.52756083  0.66431034 -0.05081236  0.42593611
  0.26701995  1.45889113  1.22051689  0.10810379 -0.84539315  1.29997497
  0.34647803 -0.36864468 -1.08376738 -1.16322546 -1.24268354 -0.2891866
 -1.87834817 -1.32214162 -1.79889009 -1.63997393 -1.56051586 -1.71943201
 -2.03726433 -1.95780625 -1.4015997  -2.19618048]
AveRooms [ 0.62855945  0.32704136  1.15562047 ... -0.09031802 -0.04021111
 -0.07044252]
AveBedrms [-0.15375759 -0.26333577 -0.04901636 ...  0.10884307  0.15877763
  0.1384028 ]
Population [-0.9744286   0.86143887 -0.82077735 ...  1.44337088  1.13165313
  4.84489156]
AveOcc

Dependecies for MAR & MNAR

In [14]:
mean_medinc = data["MedInc"].mean()
median_houseage = data["HouseAge"].median()
mean_averooms = data["AveRooms"].mean()
median_avebedrms = data["AveBedrms"].median()
mean_population = data["Population"].mean()
median_aveoccup = data["AveOccup"].median()
mean_latitude = data["Latitude"].mean()
median_longitude = data["Longitude"].median()

dependencies_mar = {
    "HouseAge": {
        "influencers": ["MedInc"],
        "condition": lambda row: True,
        "probability": lambda row: 0.5 if row["MedInc"] > mean_medinc else 0.1
    },
    "AveRooms": {
        "influencers": ["HouseAge"],
        "condition": lambda row: True,
        "probability": lambda row: 0.5 if row["HouseAge"] > median_houseage else 0.1
    },
    "AveBedrms": {
        "influencers": ["AveRooms"],
        "condition": lambda row: True,
        "probability": lambda row: 0.5 if row["AveRooms"] < mean_averooms else 0.1
    },
    "Population": {
        "influencers": ["AveOccup"],
        "condition": lambda row: True,
        "probability": lambda row: 0.5 if row["AveOccup"] > median_aveoccup else 0.1
    },
    "AveOccup": {
        "influencers": ["Population"],
        "condition": lambda row: True,
        "probability": lambda row: 0.5 if row["Population"] < mean_population else 0.1
    },
    "Latitude": {
        "influencers": ["AveRooms"],
        "condition": lambda row: True,
        "probability": lambda row: 0.5 if row["AveRooms"] > mean_averooms else 0.1
    },
    "Longitude": {
        "influencers": ["Latitude"],
        "condition": lambda row: True,
        "probability": lambda row: 0.5 if row["Latitude"] < mean_latitude else 0.1
    },
    "MedInc": {
        "influencers": ["Longitude"],
        "condition": lambda row: True,
        "probability": lambda row: 0.5 if row["Longitude"] > median_longitude else 0.1
    }
}


In [15]:
dependencies_mnar = {
    "MedInc": {
        "condition": lambda row: True,
        "probability": lambda row: 0.5 if row["MedInc"] > mean_medinc else 0.1
    },
    "HouseAge": {
        "condition": lambda row: True,
        "probability": lambda row: 0.5 if row["HouseAge"] < median_houseage else 0.1
    },
    "AveRooms": {
        "condition": lambda row: True,
        "probability": lambda row: 0.5 if row["AveRooms"] > mean_averooms else 0.1
    },
    "AveBedrms": {
        "condition": lambda row: True,
        "probability": lambda row: 0.5 if row["AveBedrms"] < median_avebedrms else 0.1
    },
    "Population": {
        "condition": lambda row: True,
        "probability": lambda row: 0.5 if row["Population"] > mean_population else 0.1
    },
    "AveOccup": {
        "condition": lambda row: True,
        "probability": lambda row: 0.5 if row["AveOccup"] > median_aveoccup else 0.1
    },
    "Latitude": {
        "condition": lambda row: True,
        "probability": lambda row: 0.5 if row["Latitude"] > mean_latitude else 0.1
    },
    "Longitude": {
        "condition": lambda row: True,
        "probability": lambda row: 0.5 if row["Longitude"] < median_longitude else 0.1
    }
}


ECE VS Model Iterations and MAE vs Model Iterations and time vs Model Iterations

In [16]:
import numpy as np
import torch


def calibration_curve(true_values, imputed_mean, imputed_std):
    imputed_std_safe = imputed_std + 1e-6
    cdf_vals = norm.cdf(true_values, loc=imputed_mean, scale=imputed_std_safe)
    quantiles = np.linspace(0, 1, 11)
    proportions = [(cdf_vals < q).mean() for q in quantiles]
    return quantiles, proportions

def compute_ece(true_values, imputed_mean, imputed_std):
    quantiles, proportions = calibration_curve(true_values, imputed_mean, imputed_std)
    ece = np.abs(np.array(proportions) - quantiles).mean()
    return ece

def run_gain_s(X):
    imputer = GAINImputer(epochs=100, batch_size=128)
    imputer.fit_transform(X)           # train and impute once
    return imputer.sample(num_samples=50)  # return mean, std


missing_rate = 30
runs = 5  # for stochastic methods
iterations = [100,300,500,700,900]

rng = np.random.default_rng(42)

# Generate MCAR missing data
generator_mcar = Inject_Missing_Values()
miss_mcar, _ = generator_mcar.MCAR(X, missing_rate=missing_rate)
missing_mask = miss_mcar.isna()
mask_np = missing_mask.values.astype(bool)
X_missing_tensor = torch.tensor(miss_mcar.values, dtype=torch.float64)

results_mcar_ece = []
results_mcar_mae = []

for itr in iterations:
    # Define all methods
    methods = {
        "OT-Impute": lambda Y: OTimputer(niter=itr, batchsize=128).fit_transform(Y),
        "MICE": lambda Y: IterativeImputer(max_iter=itr, sample_posterior=True).fit_transform(Y),
        "MIWAE": lambda Y: MIWAEImputer(input_dim=Y.shape[1], latent_dim=10, hidden_dims=[128,64], K=20, epochs=itr).fit_transform(Y),
        "GAIN": lambda Y: GAINImputer(epochs=itr, batch_size=128).fit_transform(Y),
    
    }

    # Methods that need multiple stochastic runs
    multi_run_methods = ["OT-Impute", "MICE", "MIWAE", "GAIN"]
    
    for method_name, impute_func in methods.items():
        print(f"\nRunning {method_name}...")
        imputed_runs = []
        t_method_start = time.perf_counter()
        for i in range(runs):
            print(f"Run {i+1}...")

            # Select input type
            input_data = X_missing_tensor if method_name in ["OT-Impute", "GAIN"] else miss_mcar

            
            result = impute_func(input_data)
            

            # Handle methods returning (mean, std)
            if isinstance(result, tuple):
                X_imputed, X_std = result
            else:
                X_imputed = result
                if torch.is_tensor(X_imputed):
                    X_std = np.zeros_like(X_imputed.detach())  # placeholder
                else:
                    X_std = np.zeros_like(X_imputed)

            # Convert tensors to numpy
            if torch.is_tensor(X_imputed):
                X_imputed = X_imputed.detach().numpy()
            if torch.is_tensor(X_std):
                X_std = X_std.detach().numpy()

            imputed_runs.append((X_imputed, X_std))
        
        total_time_sec = time.perf_counter() - t_method_start
        imputed_stack = np.stack([m[0] for m in imputed_runs], axis=0)
        imputed_stack_std = np.stack([m[1] for m in imputed_runs], axis=0)
        imputed_mean = np.nanmean(imputed_stack, axis=0)
        imputed_std = np.nanstd(imputed_stack, axis=0)
        

        # Compute ECE for highest and lowest entropy attributes
        ece = compute_ece(
            GroundTruth.values[missing_mask.values],
            imputed_mean[missing_mask.values],
            imputed_std[missing_mask.values]
        )

            
        mae = mean_absolute_error(
        GroundTruth.values[missing_mask.values],
        imputed_mean[missing_mask.values]
    )
            
            
        results_mcar_ece.append({
                "Method": method_name,
                "MissingRate": missing_rate,
                "MissingType": "MCAR",
                "Iteration":itr,
                "ECE": ece,
                "Time":  total_time_sec
                

            })
        
        results_mcar_mae.append({
            "Method": method_name,
            "MissingRate": missing_rate,
            "MissingType": "MCAR",
            "Iteration":itr,
            "MAE": mae,
            "Time":  total_time_sec

        })
df_mae = pd.DataFrame(results_mcar_mae)

# Save to CSV
df_mae.to_csv(r"C:\Users\zhossai3\Desktop\Uncertainty\imputation_Uncertainty\Output\results\Housing_mcar_mae.csv", index=False) 
df_ece = pd.DataFrame(results_mcar_ece)

# Save to CSV
df_ece.to_csv(r"C:\Users\zhossai3\Desktop\Uncertainty\imputation_Uncertainty\Output\results\Housing_mcar_ece.csv", index=False) 


Running OT-Impute...
Run 1...
Run 2...
Run 3...
Run 4...
Run 5...

Running MICE...
Run 1...
Run 2...
Run 3...
Run 4...
Run 5...

Running MIWAE...
Run 1...


KeyboardInterrupt: 

## Visualizations

ECE vs Iterations

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm

# ===== Load CSV results (change path if needed) =====
df_results = pd.read_csv(
    r"C:\Users\zhossai3\Desktop\Uncertainty\imputation_Uncertainty\Output\results\Housing_mcar_ece.csv"
    # e.g., could also be your Energy file if it contains Iteration info
)

# Normalize column names if needed
if "Algorithm" not in df_results.columns and "Method" in df_results.columns:
    df_results = df_results.rename(columns={"Method": "Algorithm"})

# Keep MCAR only if present (optional)
if "MissingType" in df_results.columns:
    df_results = df_results[df_results["MissingType"] == "MCAR"].copy()

# Ensure numeric Iteration
if "Iteration" in df_results.columns:
    df_results["Iteration"] = pd.to_numeric(df_results["Iteration"], errors="coerce")

# --- If ECE isn't present, compute it per (Algorithm, Iteration) from per-point data ---
def compute_ece(true_vals, mean_vals, std_vals):
    std_safe = np.asarray(std_vals) + 1e-6
    cdf_vals = norm.cdf(np.asarray(true_vals), loc=np.asarray(mean_vals), scale=std_safe)
    qs = np.linspace(0, 1, 11)
    props = [(cdf_vals < q).mean() for q in qs]
    return float(np.abs(np.array(props) - qs).mean())

if "ECE" not in df_results.columns:
    required = {"Algorithm", "Iteration", "TrueValue", "ImputedMean", "ImputedStd"}
    if required.issubset(df_results.columns):
        ece_rows = []
        for (algo, itr), g in df_results.groupby(["Algorithm", "Iteration"]):
            ece_rows.append({
                "Algorithm": algo,
                "Iteration": itr,
                "ECE": compute_ece(g["TrueValue"].values, g["ImputedMean"].values, g["ImputedStd"].values),
            })
        df_results = pd.DataFrame(ece_rows)
    else:
        raise ValueError("CSV must contain ECE or per-point columns (TrueValue, ImputedMean, ImputedStd).")

# ---- Consistent Plot Styling ----
x_label_fontsize = 25
y_label_fontsize = 25
x_tick_fontsize = 23
y_tick_fontsize = 23
legend_title_fontsize = 22
legend_label_fontsize = 20
marker_size = 15

plt.rcParams.update({
    'axes.labelsize': y_label_fontsize,
    'xtick.labelsize': x_tick_fontsize,
    'ytick.labelsize': y_tick_fontsize,
    'legend.fontsize': legend_label_fontsize,
    'legend.title_fontsize': legend_title_fontsize
})

sns.set(style="whitegrid", font_scale=1.2)

# Define consistent colors and markers
algo_styles = {
    "OT-Impute": {"color": "#1f77b4", "marker": "o"},   # Blue circle
    "MICE":      {"color": "#ff7f0e", "marker": "s"},   # Orange square
    "MIWAE":     {"color": "#27d62d", "marker": "D"},   # Green diamond
    "GAIN":      {"color": "#8e20f5", "marker": "v"},   # Purple triangle-down
    "MIWAE-S": {"color": "#92d7aa", "marker": "D"},
    "GAIN-S":  {"color": "#b89dd2", "marker": "v"},
    "MIWAE-U": {"color": "#308149", "marker": "D"},
    "GAIN-U":  {"color": "#431271", "marker": "v"},
}

algorithms = df_results["Algorithm"].unique()

# ---- Plot: ECE vs Iteration (one line per algorithm) ----
plt.figure(figsize=(8, 8))
for algo in algorithms:
    # Skip S/U variants if you don't want them
    if algo in {"GAIN-S", "GAIN-U", "MIWAE-U", "MIWAE-S"}:
        continue

    g = df_results[df_results["Algorithm"] == algo].sort_values("Iteration")
    if g.empty:
        continue

    style = algo_styles.get(algo, {"color": "black", "marker": "o"})
    plt.plot(
        g["Iteration"], g["ECE"],
        marker=style["marker"], color=style["color"],
        markersize=marker_size, linewidth=2,
        label=algo,
        markeredgecolor="black", markeredgewidth=1.2
    )

# Axis labels
plt.xlabel("Iterations", fontsize=x_label_fontsize)
plt.ylabel("ECE", fontsize=y_label_fontsize)

# Legend
plt.legend(title="Imputation Methods")

# Borders (black edges)
ax = plt.gca()
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color('black')
    spine.set_linewidth(2)

# Grid lines
ax.yaxis.grid(True, linestyle='--', color='grey', linewidth=0.7, alpha=0.7)

# Tick params
ax.tick_params(axis='x', labelsize=x_tick_fontsize)
ax.tick_params(axis='y', labelsize=y_tick_fontsize)
ax.yaxis.set_tick_params(width=1.5, length=5)

plt.tight_layout()
save_path = r"C:\Users\zhossai3\Desktop\Uncertainty\imputation_Uncertainty\Output\Housing_MCAR30_ECE_vs_Iteration.pdf"
plt.savefig(save_path, format="pdf", bbox_inches="tight")
plt.show()

print("Saved to:", save_path)


MAE VS Iterations

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm

# ===== Load CSV results (change path if needed) =====
df_results = pd.read_csv(
    r"C:\Users\zhossai3\Desktop\Uncertainty\imputation_Uncertainty\Output\results\Housing_mcar_mae.csv"
    # e.g., could also be your Energy file if it contains Iteration info
)

# Normalize column names if needed
if "Algorithm" not in df_results.columns and "Method" in df_results.columns:
    df_results = df_results.rename(columns={"Method": "Algorithm"})

# Keep MCAR only if present (optional)
if "MissingType" in df_results.columns:
    df_results = df_results[df_results["MissingType"] == "MCAR"].copy()

# Ensure numeric Iteration
if "Iteration" in df_results.columns:
    df_results["Iteration"] = pd.to_numeric(df_results["Iteration"], errors="coerce")



# ---- Consistent Plot Styling ----
x_label_fontsize = 25
y_label_fontsize = 25
x_tick_fontsize = 23
y_tick_fontsize = 23
legend_title_fontsize = 22
legend_label_fontsize = 20
marker_size = 15

plt.rcParams.update({
    'axes.labelsize': y_label_fontsize,
    'xtick.labelsize': x_tick_fontsize,
    'ytick.labelsize': y_tick_fontsize,
    'legend.fontsize': legend_label_fontsize,
    'legend.title_fontsize': legend_title_fontsize
})

sns.set(style="whitegrid", font_scale=1.2)

# Define consistent colors and markers
algo_styles = {
    "OT-Impute": {"color": "#1f77b4", "marker": "o"},   # Blue circle
    "MICE":      {"color": "#ff7f0e", "marker": "s"},   # Orange square
    "MIWAE":     {"color": "#27d62d", "marker": "D"},   # Green diamond
    "GAIN":      {"color": "#8e20f5", "marker": "v"},   # Purple triangle-down
    "MIWAE-S": {"color": "#92d7aa", "marker": "D"},
    "GAIN-S":  {"color": "#b89dd2", "marker": "v"},
    "MIWAE-U": {"color": "#308149", "marker": "D"},
    "GAIN-U":  {"color": "#431271", "marker": "v"},
}

algorithms = df_results["Algorithm"].unique()

# ---- Plot: ECE vs Iteration (one line per algorithm) ----
plt.figure(figsize=(8, 8))
for algo in algorithms:
    # Skip S/U variants if you don't want them
    if algo in {"GAIN-S", "GAIN-U", "MIWAE-U", "MIWAE-S"}:
        continue

    g = df_results[df_results["Algorithm"] == algo].sort_values("Iteration")
    if g.empty:
        continue

    style = algo_styles.get(algo, {"color": "black", "marker": "o"})
    plt.plot(
        g["Iteration"], g["MAE"],
        marker=style["marker"], color=style["color"],
        markersize=marker_size, linewidth=2,
        label=algo,
        markeredgecolor="black", markeredgewidth=1.2
    )

# Axis labels
plt.xlabel("Iterations", fontsize=x_label_fontsize)
plt.ylabel("MAE", fontsize=y_label_fontsize)

# Legend
plt.legend(title="Imputation Methods")

# Borders (black edges)
ax = plt.gca()
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color('black')
    spine.set_linewidth(2)

# Grid lines
ax.yaxis.grid(True, linestyle='--', color='grey', linewidth=0.7, alpha=0.7)

# Tick params
ax.tick_params(axis='x', labelsize=x_tick_fontsize)
ax.tick_params(axis='y', labelsize=y_tick_fontsize)
ax.yaxis.set_tick_params(width=1.5, length=5)

plt.tight_layout()
save_path = r"C:\Users\zhossai3\Desktop\Uncertainty\imputation_Uncertainty\Output\Housing_MCAR30_MAE_vs_Iteration.pdf"
plt.savefig(save_path, format="pdf", bbox_inches="tight")
plt.show()

print("Saved to:", save_path)


In [ ]:

df_results = pd.read_csv(
    r"C:\Users\zhossai3\Desktop\Uncertainty\imputation_Uncertainty\Output\results\Housing_mcar_ece.csv"
    # ^ change to your file if needed; any CSV with 'Iteration' and 'Time' works
)

# Normalize column names if needed
if "Algorithm" not in df_results.columns and "Method" in df_results.columns:
    df_results = df_results.rename(columns={"Method": "Algorithm"})

# (Optional) keep MCAR only if present
if "MissingType" in df_results.columns:
    df_results = df_results[df_results["MissingType"] == "MCAR"].copy()

# Ensure numeric
df_results["Iteration"] = pd.to_numeric(df_results["Iteration"], errors="coerce")
df_results["Time"] = pd.to_numeric(df_results["Time"], errors="coerce")
df_results = df_results.dropna(subset=["Iteration", "Time"])

# If duplicates exist for (Algorithm, Iteration), keep the last one (or change to 'mean')
df_results = (
    df_results.sort_values(["Algorithm", "Iteration"])
              .drop_duplicates(subset=["Algorithm", "Iteration"], keep="last")
)

algorithms = df_results["Algorithm"].unique()

# ---- Consistent Plot Styling ----
x_label_fontsize = 25
y_label_fontsize = 25
x_tick_fontsize = 23
y_tick_fontsize = 23
legend_title_fontsize = 22
legend_label_fontsize = 20
marker_size = 15

plt.rcParams.update({
    'axes.labelsize': y_label_fontsize,
    'xtick.labelsize': x_tick_fontsize,
    'ytick.labelsize': y_tick_fontsize,
    'legend.fontsize': legend_label_fontsize,
    'legend.title_fontsize': legend_title_fontsize
})

sns.set(style="whitegrid", font_scale=1.2)

# Define consistent colors and markers
algo_styles = {
    "OT-Impute": {"color": "#1f77b4", "marker": "o"},   # Blue circle
    "MICE":      {"color": "#ff7f0e", "marker": "s"},   # Orange square
    "MIWAE":     {"color": "#27d62d", "marker": "D"},   # Green diamond
    "GAIN":      {"color": "#8e20f5", "marker": "v"},   # Purple triangle-down
    "MIWAE-S": {"color": "#92d7aa", "marker": "D"},
    "GAIN-S":  {"color": "#b89dd2", "marker": "v"},
    "MIWAE-U": {"color": "#308149", "marker": "D"},
    "GAIN-U":  {"color": "#431271", "marker": "v"},
}

# ---- Plot: Time vs Iteration ----
plt.figure(figsize=(8, 8))
ax = plt.gca()

for algo in algorithms:
    # Skip S/U variants if you don't want them (match your previous plot)
    if algo in {"GAIN-S", "GAIN-U", "MIWAE-U", "MIWAE-S"}:
        continue

    g = df_results[df_results["Algorithm"] == algo].sort_values("Iteration")
    if g.empty:
        continue

    style = algo_styles.get(algo, {"color": "black", "marker": "o"})
    ax.plot(
        g["Iteration"], g["Time"],
        marker=style["marker"], color=style["color"],
        markersize=marker_size, linewidth=2,
        label=algo,
        markeredgecolor="black", markeredgewidth=1.2
    )

# Axis labels
ax.set_xlabel("Iterations", fontsize=x_label_fontsize)
ax.set_ylabel("Time (s)", fontsize=y_label_fontsize)

# Optional: use log scale if times vary a lot
# ax.set_yscale("log")

# Legend
ax.legend(title="Imputation Methods")

# Borders (black edges)
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color('black')
    spine.set_linewidth(2)

# Grid lines & ticks
ax.yaxis.grid(True, linestyle='--', color='grey', linewidth=0.7, alpha=0.7)
ax.tick_params(axis='x', labelsize=x_tick_fontsize)
ax.tick_params(axis='y', labelsize=y_tick_fontsize)
ax.yaxis.set_tick_params(width=1.5, length=5)

plt.tight_layout()
save_path = r"C:\Users\zhossai3\Desktop\Uncertainty\imputation_Uncertainty\Output\Housing_MCAR30_Time_vs_Iteration.pdf"
plt.savefig(save_path, format="pdf", bbox_inches="tight")
plt.show()

print("Saved to:", save_path)


time vs sample

In [18]:

missing_rates = [30]  # 10% to 90% missing


rng = np.random.default_rng(42)
samples = [20,40,60,80,100]

# Create nested results dictionary
results_mcar = []




generator_mcar = Inject_Missing_Values()
miss_mcar, index_mcar = generator_mcar.MCAR(X,missing_rate=missing_rate)
X_missing_tensor = torch.tensor(miss_mcar.values, dtype=torch.float64)
       
missing_mask = miss_mcar.isna()

    
for sample in samples:  
    print(sample) 

    print("MIWAE-S")   
    if torch.cuda.is_available(): torch.cuda.synchronize()            # optional for GPU accuracy
    t0 = time.perf_counter()
    miwae = MIWAEImputer(input_dim=miss_mcar.shape[1], latent_dim=10,
                            hidden_dims=[128, 64], K=20, epochs=100)  
    miwae = MIWAEImputer(input_dim=miss_mcar.shape[1], latent_dim=10,
                            hidden_dims=[128, 64], K=20, epochs=100)
    mean_miwae, std_miwae= miwae.fit_transform_std(miss_mcar, return_std=True, inference_K=sample)
    miwae_time = time.perf_counter() - t0

    for t, m, s in zip(GroundTruth.values[missing_mask.values].flatten(),
                            mean_miwae[missing_mask.values].flatten(),
                            std_miwae[missing_mask.values].flatten()):
        results_mcar.append({
            "Algorithm": "MIWAE-S",
                   
            "MissingRate": missing_rate,
            "Time":  miwae_time,
            "TrueValue": t,
            "ImputedMean": m,
            "ImputedStd": s,
            "Sample": sample
                    
                })

            #GAIN
    print("GAIN-S")
    if torch.cuda.is_available(): torch.cuda.synchronize()            # optional for GPU accuracy
    t0 = time.perf_counter()    
    imputer_gain = GAINImputer(epochs=100, batch_size=128)
    X_imputed_gain = imputer_gain.fit_transform(X_missing_tensor)
    mean_gain, std_gain = imputer_gain.sample(num_samples=sample)
    gain_time = time.perf_counter() - t0
            
    for t, m, s in zip(GroundTruth.values[missing_mask.values].flatten(),
                            mean_gain[missing_mask.values].flatten(),
                            std_gain[missing_mask.values].flatten()):
        results_mcar.append({
                        "Algorithm": "GAIN-S",
                        "MissingRate": missing_rate,
                        "Time": gain_time,
                        "TrueValue": t,
                        "ImputedMean": m,
                        "ImputedStd": s,
                        "Sample": sample
                    })
        



df_results = pd.DataFrame(results_mcar)
df_results.to_csv(r"C:\Users\zhossai3\Desktop\Uncertainty\imputation_Uncertainty\Output\results\Housing_mcar_timeVSsamples.csv", index=False)

20
MIWAE-S


KeyboardInterrupt: 

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ===== Load CSV with per-cell rows (has TimeSec, Sample, Algorithm) =====
df_results = pd.read_csv(
    r"C:\Users\zhossai3\Desktop\Uncertainty\imputation_Uncertainty\Output\results\Housing_mcar_timeVSsamples.csv"
)

# If you saved a compact summary instead, you can load it directly:
# df_results = pd.read_csv(
#     r"C:\Users\zhossai3\Desktop\Uncertainty\imputation_Uncertainty\Output\results\Wine_mcar_time_summary.csv"
# )

# Normalize column names if needed
if "Algorithm" not in df_results.columns and "Method" in df_results.columns:
    df_results = df_results.rename(columns={"Method": "Algorithm"})

# Keep only needed cols and ensure numeric Sample/Time
keep_cols = {"Algorithm", "Sample", "Time"}
if not keep_cols.issubset(df_results.columns):
    raise ValueError(f"CSV must contain columns: {keep_cols}")
df = df_results[list(keep_cols)].copy()
df["Sample"] = pd.to_numeric(df["Sample"], errors="coerce")
df["Time"] = pd.to_numeric(df["Time"], errors="coerce")
df = df.dropna(subset=["Sample", "Time"])

# Collapse per-cell duplicates → one time per (Algorithm, Sample)
# (You recorded the same TimeSec for all cells of a method/sample, so 'first' is fine.)
df_time = (
    df.sort_values(["Algorithm", "Sample"])
      .drop_duplicates(subset=["Algorithm", "Sample"], keep="first")
)

algorithms = df_time["Algorithm"].unique()

# ---- Consistent Plot Styling ----
x_label_fontsize = 25
y_label_fontsize = 25
x_tick_fontsize = 23
y_tick_fontsize = 23
legend_title_fontsize = 22
legend_label_fontsize = 20
marker_size = 15

plt.rcParams.update({
    'axes.labelsize': y_label_fontsize,
    'xtick.labelsize': x_tick_fontsize,
    'ytick.labelsize': y_tick_fontsize,
    'legend.fontsize': legend_label_fontsize,
    'legend.title_fontsize': legend_title_fontsize
})

sns.set(style="whitegrid", font_scale=1.2)

# Define consistent colors and markers
algo_styles = {
    "OT-Impute": {"color": "#1f77b4", "marker": "o"},   # Blue circle
    "MICE":      {"color": "#ff7f0e", "marker": "s"},   # Orange square
    "MIWAE":     {"color": "#27d62d", "marker": "D"},   # Green diamond
    "GAIN":      {"color": "#8e20f5", "marker": "v"},   # Purple triangle-down
    "MIWAE-S": {"color": "#92d7aa", "marker": "D"},
    "GAIN-S":  {"color": "#b89dd2", "marker": "v"},
    "MIWAE-U": {"color": "#308149", "marker": "D"},
    "GAIN-U":  {"color": "#431271", "marker": "v"},
}

# ---- Plot: Time vs Sample ----
plt.figure(figsize=(8, 8))
ax = plt.gca()

for algo in algorithms:
    g = df_time[df_time["Algorithm"] == algo].sort_values("Sample")
    if g.empty:
        continue

    style = algo_styles.get(algo, {"color": "black", "marker": "o"})
    ax.plot(
        g["Sample"], g["Time"],
        marker=style["marker"], color=style["color"],
        markersize=marker_size, linewidth=2,
        label=algo,
        markeredgecolor="black", markeredgewidth=1.2
    )

# Axis labels
ax.set_xlabel("Samples", fontsize=x_label_fontsize)
ax.set_ylabel("Time (s)", fontsize=y_label_fontsize)

# Optional: if times vary a lot, use log scale
# ax.set_yscale("log")

# Legend
ax.legend(title="Imputation Methods")

# Borders (black edges)
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color('black')
    spine.set_linewidth(2)

# Grid & ticks
ax.yaxis.grid(True, linestyle='--', color='grey', linewidth=0.7, alpha=0.7)
ax.tick_params(axis='x', labelsize=x_tick_fontsize)
ax.tick_params(axis='y', labelsize=y_tick_fontsize)
ax.yaxis.set_tick_params(width=1.5, length=5)

plt.tight_layout()
save_path = r"C:\Users\zhossai3\Desktop\Uncertainty\imputation_Uncertainty\Output\Housing_MCAR30_Time_vs_Sample_allMethods.pdf"
plt.savefig(save_path, format="pdf", bbox_inches="tight")
plt.show()

print("Saved to:", save_path)


In [16]:
import numpy as np
import torch


def calibration_curve(true_values, imputed_mean, imputed_std):
    imputed_std_safe = imputed_std + 1e-6
    cdf_vals = norm.cdf(true_values, loc=imputed_mean, scale=imputed_std_safe)
    quantiles = np.linspace(0, 1, 11)
    proportions = [(cdf_vals < q).mean() for q in quantiles]
    return quantiles, proportions

def compute_ece(true_values, imputed_mean, imputed_std):
    quantiles, proportions = calibration_curve(true_values, imputed_mean, imputed_std)
    ece = np.abs(np.array(proportions) - quantiles).mean()
    return ece

def run_gain_s(X):
    imputer = GAINImputer(epochs=100, batch_size=128)
    imputer.fit_transform(X)           # train and impute once
    return imputer.sample(num_samples=50)  # return mean, std


missing_rate = 30
runs = 2  # for stochastic methods


rng = np.random.default_rng(42)

# Generate MCAR missing data
generator_mcar = Inject_Missing_Values()
miss_mcar, _ = generator_mcar.MCAR(X, missing_rate=missing_rate)
missing_mask = miss_mcar.isna()
mask_np = missing_mask.values.astype(bool)
X_missing_tensor = torch.tensor(miss_mcar.values, dtype=torch.float64)

results_mcar_ece = []
results_mcar_mae = []


    # Define all methods
methods = {
        "OT-Impute": lambda Y: OTimputer(niter=300, batchsize=128).fit_transform(Y),
        "MICE": lambda Y: IterativeImputer(max_iter=100, sample_posterior=True).fit_transform(Y),
        "MIWAE": lambda Y: MIWAEImputer(input_dim=Y.shape[1], latent_dim=10, hidden_dims=[128,64], K=20, epochs=5000).fit_transform(Y),
        "GAIN": lambda Y: GAINImputer(epochs=5000, batch_size=128).fit_transform(Y),
    }

    # Methods that need multiple stochastic runs
multi_run_methods = ["OT-Impute", "MICE", "MIWAE", "GAIN"]
    
for method_name, impute_func in methods.items():
        print(f"\nRunning {method_name}...")
        imputed_runs = []
        t_method_start = time.perf_counter()
        for i in range(runs):
            print(f"Run {i+1}...")

            # Select input type
            input_data = X_missing_tensor if method_name in ["OT-Impute", "GAIN"] else miss_mcar

            
            result = impute_func(input_data)
            

            # Handle methods returning (mean, std)
            if isinstance(result, tuple):
                X_imputed, X_std = result
            else:
                X_imputed = result
                if torch.is_tensor(X_imputed):
                    X_std = np.zeros_like(X_imputed.detach())  # placeholder
                else:
                    X_std = np.zeros_like(X_imputed)

            # Convert tensors to numpy
            if torch.is_tensor(X_imputed):
                X_imputed = X_imputed.detach().numpy()
            if torch.is_tensor(X_std):
                X_std = X_std.detach().numpy()

            imputed_runs.append((X_imputed, X_std))
        
        total_time_sec = time.perf_counter() - t_method_start
        imputed_stack = np.stack([m[0] for m in imputed_runs], axis=0)
        imputed_stack_std = np.stack([m[1] for m in imputed_runs], axis=0)
        imputed_mean = np.nanmean(imputed_stack, axis=0)
        imputed_std = np.nanstd(imputed_stack, axis=0)
        

        # Compute ECE for highest and lowest entropy attributes
        ece = compute_ece(
            GroundTruth.values[missing_mask.values],
            imputed_mean[missing_mask.values],
            imputed_std[missing_mask.values]
        )

            
        mae = mean_absolute_error(
        GroundTruth.values[missing_mask.values],
        imputed_mean[missing_mask.values]
    )
            
            
        results_mcar_ece.append({
                "Method": method_name,
                "MissingRate": missing_rate,
                "MissingType": "MCAR",
             
                "ECE": ece,
                "Time":  total_time_sec
                

            })
        
        results_mcar_mae.append({
            "Method": method_name,
            "MissingRate": missing_rate,
            "MissingType": "MCAR",
         
            "MAE": mae,
            "Time":  total_time_sec

        })
df_mae = pd.DataFrame(results_mcar_mae)

# Save to CSV
df_mae.to_csv(r"C:\Users\zhossai3\Desktop\Uncertainty\imputation_Uncertainty\Output\results\Housing_mcar_mae_test.csv", index=False) 
df_ece = pd.DataFrame(results_mcar_ece)

# Save to CSV
df_ece.to_csv(r"C:\Users\zhossai3\Desktop\Uncertainty\imputation_Uncertainty\Output\results\Housing_mcar_ece_test.csv", index=False) 


Running OT-Impute...
Run 1...
Run 2...

Running MICE...
Run 1...
Run 2...

Running MIWAE...
Run 1...


KeyboardInterrupt: 